**Installing Docling for extracting content from PDF**

In [ ]:
pip install -q docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.7/286.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 48.3 MB/s eta 0:00:00
   ━━

**Converting the document into DocLing Document so the structure is well maintained**

In [ ]:
from docling.document_converter import DocumentConverter

source = "/content/mrag-research-paper"
converter = DocumentConverter()
doc = converter.convert(source).document

[INFO] 2026-06-08 19:28:29,381 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-08 19:28:29,383 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-08 19:28:29,484 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-08 19:28:29,487 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-08 19:28:29,962 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-08 19:28:29,965 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-08 19:28:29,973 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-08 19:28:29,977 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-08 19:28:30,149 [Ra

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

* **Using HierarchicalChunker for chunking the text block**
* **This makes sure that chunking maintains the structure**
* **Storing chunks into output/chunks.json**
* **Key is sha-256 hash and value is text chunk**

In [ ]:
import json
import hashlib
from pathlib import Path

from docling_core.transforms.chunker import HierarchicalChunker

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Chunk document
chunker = HierarchicalChunker()
chunks = [chunk.text for chunk in chunker.chunk(doc)]

# Create JSON structure
chunk_json = {}

for chunk in chunks:
    chunk_hash = hashlib.sha256(
        chunk.encode("utf-8")
    ).hexdigest()

    chunk_json[chunk_hash] = chunk

# Save JSON
json_path = output_dir / "chunks.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        chunk_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved {len(chunks)} chunks to {json_path}")

Saved 48 chunks to output/chunks.json


In [ ]:
pip install -q PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 37.4 MB/s eta 0:00:00


In [ ]:
pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00


* **Every image is extracted and passed through a vision model**
* **Generate description for every image**
* **Generate sha-256 hash as key for every image**
* **Convert image into base64**
* **Save hash as key and value as [base64 string of image, description]**
* **Output/image_descriptions.json**

In [ ]:
import fitz  # PyMuPDF
import json
import base64
import hashlib
import io
import os
from pathlib import Path
from PIL import Image
from groq import Groq


# ---------------- CONFIG ----------------
PDF_PATH = "/content/mrag-research-paper"   # add .pdf if needed
OUTPUT_DIR = Path("output")
IMAGE_DIR = OUTPUT_DIR / "images"
JSON_PATH = OUTPUT_DIR / "image_descriptions.json"

OUTPUT_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

client = Groq(api_key="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")


# ---------------- HELPERS ----------------
def compress_image_to_base64(image_bytes, max_size=1024, quality=70):
    image = Image.open(io.BytesIO(image_bytes))

    if image.mode != "RGB":
        image = image.convert("RGB")

    image.thumbnail((max_size, max_size))

    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality, optimize=True)

    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def generate_description(image_base64, nearby_text):
    prompt = f"""
You are extracting information from a PDF.

Nearby text:
{nearby_text[:1000]}

Task:
Generate a concise, information-dense description of the image.
Use nearby text only when relevant.
Return only the description.
"""

    completion = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_base64}"
                        },
                    },
                ],
            }
        ],
        temperature=0.2,
        max_completion_tokens=300,
    )

    return completion.choices[0].message.content


# ---------------- MAIN ----------------
final_json = {}

pdf = fitz.open(PDF_PATH)

for page_num, page in enumerate(pdf, start=1):
    nearby_text = page.get_text("text").strip()
    images = page.get_images(full=True)

    for img_index, img in enumerate(images, start=1):
        xref = img[0]
        base_image = pdf.extract_image(xref)

        image_bytes = base_image["image"]
        image_ext = base_image["ext"]

        sha256_hash = hashlib.sha256(image_bytes).hexdigest()

        image_path = IMAGE_DIR / f"{sha256_hash}.{image_ext}"

        with open(image_path, "wb") as f:
            f.write(image_bytes)

        try:
            image_base64 = compress_image_to_base64(image_bytes)

            description = generate_description(
                image_base64=image_base64,
                nearby_text=nearby_text
            )

            final_json[sha256_hash] = [
                image_base64,
                description
            ]

            print(f"Processed image {img_index} on page {page_num}")

        except Exception as e:
            print(f"Failed image {img_index} on page {page_num}: {e}")

pdf.close()

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(final_json, f, ensure_ascii=False, indent=2)

print(f"JSON saved at: {JSON_PATH}")

Processed image 1 on page 1
Processed image 1 on page 2
Processed image 1 on page 3
Processed image 1 on page 4
JSON saved at: output/image_descriptions.json


In [ ]:
pip install -q sentence-transformers

* **Iterate over chunks.json and image_description.json**
* **Compute cosine similarity between image description and text chunks**
* **The text chunks and images that are very similar are concatenated**
* **And their ids are put together into a tuple**
* **A new JSON file is created where keys are tuple of IDS and values are concatenated text chunks and image descriptions**
* **Saved in output/merged_chunks_images.json**

In [ ]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer, util

# -------- Paths --------
output_dir = Path("output")

chunks_path = output_dir / "chunks.json"
images_path = output_dir / "image_descriptions.json"
merged_path = output_dir / "merged_chunks_images.json"

# -------- Load JSON files --------
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks_json = json.load(f)

with open(images_path, "r", encoding="utf-8") as f:
    images_json = json.load(f)

# -------- Load embedding model --------
model = SentenceTransformer("all-MiniLM-L6-v2")

# -------- Prepare data --------
chunk_ids = list(chunks_json.keys())
chunk_texts = list(chunks_json.values())

image_ids = list(images_json.keys())
image_descriptions = [
    images_json[img_id][1] for img_id in image_ids
]

# -------- Create embeddings --------
chunk_embeddings = model.encode(chunk_texts, convert_to_tensor=True)
image_embeddings = model.encode(image_descriptions, convert_to_tensor=True)

# -------- Similarity threshold --------
SIMILARITY_THRESHOLD = 0.7

merged_json = {}

# -------- Compare every chunk with every image description --------
for i, chunk_id in enumerate(chunk_ids):
    for j, image_id in enumerate(image_ids):

        similarity = util.cos_sim(
            chunk_embeddings[i],
            image_embeddings[j]
        ).item()

        if similarity >= SIMILARITY_THRESHOLD:
            key = f"({chunk_id}, {image_id})"

            merged_json[key] = {
                "chunk_id": chunk_id,
                "image_id": image_id,
                "similarity": similarity,
                "content": chunks_json[chunk_id] + "\n\n" + image_descriptions[j]
            }

# -------- Save new JSON --------
with open(merged_path, "w", encoding="utf-8") as f:
    json.dump(merged_json, f, ensure_ascii=False, indent=2)

print(f"Saved {len(merged_json)} merged records to {merged_path}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saved 9 merged records to output/merged_chunks_images.json


In [ ]:
pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

* **All the chunks in the chunks.json file are converted into embeddings**
* **All the text blocks in merged_chunks_image.json are converted into embeddings**
* **All the embeddings are stored into vector database along with indivisual IDs and tuple of IDs**

In [ ]:
import json
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer


# -------- Paths --------
output_dir = Path("output")

chunks_path = output_dir / "chunks.json"
merged_path = output_dir / "merged_chunks_images.json"

chroma_path = "chroma_db"


# -------- Load JSON files --------
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks_json = json.load(f)

with open(merged_path, "r", encoding="utf-8") as f:
    merged_json = json.load(f)


# -------- Embedding model --------
model = SentenceTransformer("all-MiniLM-L6-v2")


# -------- ChromaDB setup --------
client = chromadb.PersistentClient(path=chroma_path)

collection = client.get_or_create_collection(
    name="pdf_chunks"
)


# -------- Prepare records --------
ids = []
documents = []
metadatas = []

# 1. Add normal chunks
for chunk_id, chunk_text in chunks_json.items():
    ids.append(chunk_id)
    documents.append(chunk_text)
    metadatas.append({
        "type": "text_chunk",
        "chunk_id": chunk_id
    })

# 2. Add merged chunk + image descriptions
for merged_id, merged_data in merged_json.items():

    # If your merged JSON value is a dict
    if isinstance(merged_data, dict):
        text = merged_data["content"]
        chunk_id = merged_data.get("chunk_id")
        image_id = merged_data.get("image_id")
        similarity = merged_data.get("similarity")

    # If your merged JSON value is only plain text
    else:
        text = merged_data
        chunk_id = None
        image_id = None
        similarity = None

    ids.append(merged_id)
    documents.append(text)
    metadatas.append({
        "type": "merged_chunk_image",
        "chunk_id": chunk_id or "",
        "image_id": image_id or "",
        "similarity": similarity or 0
    })


# -------- Create embeddings --------
embeddings = model.encode(
    documents,
    convert_to_numpy=True
).tolist()


# -------- Save to ChromaDB --------
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Saved {len(ids)} records to ChromaDB")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saved 57 records to ChromaDB


* **Print the Vector Database Schema**

In [ ]:
data = collection.get()

import pandas as pd

df = pd.DataFrame({
    "id": data["ids"],
    "document": data["documents"],
    "metadata": data["metadatas"]
})

print(df.head())

                                                  id  \
0  95d13b2c6ad9fc864671c815af5463e243af0f614a21da...   
1  8126a8be362904e8c6ff70d47e70ff0895c72b7d9e7763...   
2  f77cf3cb2eca137b9d9baf4838bbde7fbbb80b1945e3e1...   
3  46aac00d267791d3a36b391bae0d5c76403099bcc382e0...   
4  bb56547fff11b0b829ca9913c2573b777faf34dbd02d75...   

                                            document  \
0  Zhucun Xue Zhejiang University Zhejiang, China...   
1  Figure 1: Research trajectory of the doctoral ...   
2  This paper presents a doctoral research focusi...   
3  Permission to make digital or hard copies of a...   
4                             MM'25, Dublin, Ireland   

                                            metadata  
0  {'type': 'text_chunk', 'chunk_id': '95d13b2c6a...  
1  {'chunk_id': '8126a8be362904e8c6ff70d47e70ff08...  
2  {'chunk_id': 'f77cf3cb2eca137b9d9baf4838bbde7f...  
3  {'chunk_id': '46aac00d267791d3a36b391bae0d5c76...  
4  {'type': 'text_chunk', 'chunk_id': 'bb56547fff..

* **Get the query from the user**
* **Convert the user query into vector embeddings**
* **Compute cosine similarity between user query and vectors in vector databse**
* **Get top 10 relevant chunks from the vector database**

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Load ChromaDB
client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_collection("pdf_chunks")

# Query
query = "What is the image saying about the RAG agent"

# Encode query
query_embedding = model.encode(
    query,
    convert_to_numpy=True
).tolist()

# Retrieve top 10 most similar documents
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

# Print results
for rank, (doc_id, doc, distance) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nRank: {rank}")
    print(f"ID: {doc_id}")
    print(f"Distance: {distance}")
    print(f"Document: {doc[:500]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Rank: 1
ID: 35ba05d5dbb368d0a88402f853778b5056e7798634c15e03773cdd8bea3180b7
Distance: 1.0314770936965942
Document: 3. RAGAgent (Sec. 4) . With the popularity of Artificial General Intelligence(AGI) [18], Agentic RAG [23, 29], by integrating the autonomous decision-making ability of agents and the dynamic knowledge enhancement mechanism of multimodal RAG, provides a disruptive solution for video tasks. Through dynamic task decomposition and multi-agents collaboration, Agentic RAG can more flexibly and actively schedule external tools and iteratively optimize. It may be the future solution direction that can c...

Rank: 2
ID: b956a3c5c24521a1966c69bd61d4820dbca592fa63ca8c876c3d5cd4bedfd266
Distance: 1.1948862075805664
Document: This paper systematically introduces our works on RAG-based visual understanding and generation, centered around RAG. Concurrently, we present a high-quality UltraVideo dataset and the HiVU/HiVGbenchmarks to evaluate the effectiveness of these methods. Looking a

* **Use a cross encoder reranker to rerank all the 10 reterieved documents**
* **Get final 5 documents from vector database**

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

pairs = [
    [query, doc]
    for doc in results["documents"][0]
]

scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(
        results["ids"][0],
        results["documents"][0],
        scores
    ),
    key=lambda x: x[2],
    reverse=True
)

top_3 = reranked_results[:5]

for rank, (doc_id, doc, score) in enumerate(top_3, start=1):
    print(f"\nRank: {rank}")
    print(f"ID: {doc_id}")
    print(f"Score: {score:.4f}")
    print(f"Document: {doc[:500]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Rank: 1
ID: (b956a3c5c24521a1966c69bd61d4820dbca592fa63ca8c876c3d5cd4bedfd266, 2d1f81c2be43812a962b31f4fb7734284cf62b746ba99f93b5b4034fdab97109)
Score: 3.0282
Document: This paper systematically introduces our works on RAG-based visual understanding and generation, centered around RAG. Concurrently, we present a high-quality UltraVideo dataset and the HiVU/HiVGbenchmarks to evaluate the effectiveness of these methods. Looking ahead, we plan to extend RAG to Agent RAG with greater practical applications and continue to delve into unified visual understanding and generation research oriented toward AGI. Legal, ethical and privacy implications. Even though our dat...

Rank: 2
ID: (87e17bedee2028db9113bc786cfac8e1c54a65c9e6e7b702edb96b92d6781b40, 2d1f81c2be43812a962b31f4fb7734284cf62b746ba99f93b5b4034fdab97109)
Score: 2.7798
Document: Current RAG research mostly focuses on text modality [7, 9, 12], static images [6], and tabular forms [4], overlooking the unique value of video as a multim

* **Create a set of all the ids - images and text chunks**
* **Print Them**

In [ ]:
import json
from pathlib import Path

# --------------------------
# Load JSON files
# --------------------------
output_dir = Path("output")

with open(output_dir / "chunks.json", "r", encoding="utf-8") as f:
    chunks_json = json.load(f)

with open(output_dir / "image_descriptions.json", "r", encoding="utf-8") as f:
    images_json = json.load(f)

# --------------------------
# Collect all unique IDs
# --------------------------
all_ids = set()

for doc_id, _, _ in top_3:

    # Merged ID format:
    # (chunk_hash, image_hash)
    if doc_id.startswith("(") and doc_id.endswith(")"):

        ids_str = doc_id[1:-1]  # remove '(' and ')'

        try:
            chunk_id, image_id = [
                x.strip()
                for x in ids_str.split(",", maxsplit=1)
            ]

            all_ids.add(chunk_id)
            all_ids.add(image_id)

        except ValueError:
            print(f"Could not parse merged ID: {doc_id}")

    else:
        all_ids.add(doc_id)

# --------------------------
# Retrieve matching chunks/images
# --------------------------
retrieved_chunks = {}
retrieved_images = {}

for item_id in all_ids:

    if item_id in chunks_json:
        retrieved_chunks[item_id] = chunks_json[item_id]

    if item_id in images_json:
        # [base64, description]
        retrieved_images[item_id] = {
            "base64": images_json[item_id][0],
            "description": images_json[item_id][1]
        }

# --------------------------
# Print text chunks
# --------------------------
print("\nTEXT CHUNKS")
print("=" * 100)

for chunk_id, chunk_text in retrieved_chunks.items():

    print(f"\nChunk ID:\n{chunk_id}")
    print("-" * 100)
    print(chunk_text[:1000])

# --------------------------
# Print image information
# --------------------------
print("\nIMAGES")
print("=" * 100)

for image_id, image_data in retrieved_images.items():

    print(f"\nImage ID:\n{image_id}")
    print("-" * 100)

    print("\nDescription:")
    print(image_data["description"])

    print("\nBase64 (first 200 chars):")
    print(image_data["base64"][:200] + "...")

    print("\n" + "=" * 100)


TEXT CHUNKS

Chunk ID:
87e17bedee2028db9113bc786cfac8e1c54a65c9e6e7b702edb96b92d6781b40
----------------------------------------------------------------------------------------------------
Current RAG research mostly focuses on text modality [7, 9, 12], static images [6], and tabular forms [4], overlooking the unique value of video as a multimodal knowledge carrier. Especially in tasks such as long video understanding, dynamic scene generation, and interactive video QA, how to use RAG to enhance the capabilities of Multimodal Large Language Models (MLLMs) is a question worthy of exploration, and it is also the topic mainly discussed and studied in this paper. Specifically, RAG can extract multi-source information from videos, including audio, caption, and OCR, which can retrieve direct video segments related to the query as well as text descriptions, helping us to better understand the video comprehensively and accurately. Especially in long video understanding, due to the current lim

In [ ]:
pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00


* **Create LLM instance**
* **Create a prompt with text content and images**
* **Generate a textual answer with Vison LLM**
* **Print the answer**

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    api_key="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",
    temperature=0.2,
    max_tokens=700
)

# --------------------------
# Prepare text context
# --------------------------
text_context = ""

for chunk_id, chunk_text in retrieved_chunks.items():
    text_context += f"\n\nChunk ID: {chunk_id}\n{chunk_text}"

# --------------------------
# Prepare multimodal message
# --------------------------
content = [
    {
        "type": "text",
        "text": f"""
You are answering a user query using retrieved PDF text chunks and related images.

User query:
{query}

Retrieved text chunks:
{text_context}

Instructions:
- Answer only from the provided text and images.
- Use images only when relevant.
- If the answer is not present, say that the provided context is insufficient.
- Be clear and concise.
"""
    }
]

# Add images, max 5 because Groq vision supports up to 5 images
for image_id, image_data in list(retrieved_images.items())[:5]:
    image_base64 = image_data["base64"]

    content.append(
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{image_base64}"
            }
        }
    )

# --------------------------
# Invoke Vision LLM
# --------------------------
response = llm.invoke([
    HumanMessage(content=content)
])
answer = response.content
print(response.content)

The image illustrates the capabilities of the RAG Agent, which appears to be a component of a larger system focused on video understanding and generation. 

The RAG Agent seems to play a crucial role in both video understanding and generation tasks. For video understanding, it can process queries at different levels of difficulty and retrieve relevant information from a database. For video generation, it can take a text prompt, such as "Spongebob dance ballet," and generate a video based on that prompt.

Key features of the RAG Agent include:
* Query classification: It can classify queries into different levels of difficulty, including straightforward, easy, and hard levels.
* Retrieval from database: It can retrieve relevant information from a database to aid in video understanding and generation.
* LLMs decoupling: It can decouple large language models (LLMs) to generate videos based on text prompts.

The image also highlights the RAG Agent's potential applications in various areas, 

In [ ]:
pip install -q networkx sentence-transformers

* **For multimodal answer we need text and images in the answer**
* **We are using Edmond's Blossom Algorithm**
* **Creating graph with nodes as Images and sentences of answer generated**
* **Edge weight is the cosine similarity between sentence and image**
* Create the graph**

In [ ]:
import networkx as nx
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = answer.split(". ")
image_ids = list(retrieved_images.keys())
image_descriptions = [
    retrieved_images[img_id]["description"]
    for img_id in image_ids
]

sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
image_embeddings = model.encode(image_descriptions, convert_to_tensor=True)

G = nx.Graph()

for i, sentence in enumerate(sentences):
    for j, image_id in enumerate(image_ids):
        similarity = util.cos_sim(
            sentence_embeddings[i],
            image_embeddings[j]
        ).item()

        if similarity > 0.35:
            G.add_edge(
                f"s_{i}",
                f"img_{j}",
                weight=similarity
            )

matching = nx.algorithms.matching.max_weight_matching(
    G,
    maxcardinality=False
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

* **Insert the images into sentences**

In [ ]:
sentence_to_image = {}

for a, b in matching:
    if a.startswith("s_"):
        sentence_id = int(a.split("_")[1])
        image_id = image_ids[int(b.split("_")[1])]
    else:
        sentence_id = int(b.split("_")[1])
        image_id = image_ids[int(a.split("_")[1])]

    sentence_to_image[sentence_id] = image_id

* **Generate Multimodal Answer**

In [ ]:
for i, sentence in enumerate(sentences):
    print(sentence)

    if i in sentence_to_image:
        img_id = sentence_to_image[i]
        print(f"\n[Insert image: {img_id}]\n")

The image illustrates the capabilities of the RAG Agent, which appears to be a component of a larger system focused on video understanding and generation


The RAG Agent seems to play a crucial role in both video understanding and generation tasks
For video understanding, it can process queries at different levels of difficulty and retrieve relevant information from a database
For video generation, it can take a text prompt, such as "Spongebob dance ballet," and generate a video based on that prompt.

Key features of the RAG Agent include:
* Query classification: It can classify queries into different levels of difficulty, including straightforward, easy, and hard levels.
* Retrieval from database: It can retrieve relevant information from a database to aid in video understanding and generation.
* LLMs decoupling: It can decouple large language models (LLMs) to generate videos based on text prompts.

The image also highlights the RAG Agent's potential applications in various areas, inc

In [ ]:
# https://docling-project.github.io/docling/